# Study C: a fixed held-out digit comparison

Study C constructs directions from handwritten digit images and compares a first-moment model with a richer first-and-second-moment model. Query-specific weights define the fitted and held-out comparisons.

The original partition remains fixed in both modes: six hundred images train the representation, forty form the query pool, six hundred form the fitting pool, and five hundred fifty-seven form the evaluation pool.

Smoke mode analyzes four of the frozen forty queries at each of three temperatures, giving twelve comparisons. Paper mode analyzes all forty queries at each temperature, giving one hundred twenty comparisons. Smoke mode does not shrink or reshuffle the data partition.

The notebook calls the shared workflow and reads its saved results. It does not implement a second fitting routine. Generated outputs go to `results/smoke` or `results/paper`; the frozen `reference_results` directory is not overwritten.

Run all cells from top to bottom after changing the mode. Smoke mode checks execution and output structure. Its tiny simulation samples are not evidence for the manuscript's statistical conclusions.


In [1]:
# Change MODE to "paper" to run the complete manuscript experiment.
# Publication figures require LaTeX and the configured image-conversion tools.
MODE = "smoke"
FIGURES = False
assert MODE in {"smoke", "paper"}


In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
REPO = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "workflow.py").is_file() and (p / "code" / "gid_pipeline.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Open this notebook from the repository root or its notebooks directory.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from workflow import run_study


def read_json(path):
    return json.loads(Path(path).read_text())


def require_finite(frame, columns):
    values = frame.loc[:, columns].apply(pd.to_numeric, errors="raise").to_numpy()
    assert np.isfinite(values).all(), f"Nonfinite values in {columns}"


def show_run_figure(relative_path):
    if not FIGURES:
        print("Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.")
        return
    figure = RUN / relative_path
    if not figure.is_file():
        raise FileNotFoundError(f"The requested current-run figure was not generated: {figure}")
    display(Markdown(f"Figure from `{figure.relative_to(REPO)}` ({MODE} mode)."))
    display(Image(filename=str(figure)))

print(f"Repository root located: {REPO.name}")
print(f"Mode: {MODE}; figures: {FIGURES}")


Repository root located: github
Mode: smoke; figures: False


In [3]:
RUN = Path(run_study("c", mode=MODE, figures=FIGURES)).resolve()
assert RUN == (REPO / "results" / MODE).resolve()
assert RUN != (REPO / "reference_results").resolve()
print(f"Reading generated results from {RUN.relative_to(REPO)}")


smoke: study_c_digits


smoke: study_c_validate


Reading generated results from results/smoke


## Verify the fixed split and numerical checks

Scaling and principal components use only the representation split. The query ranking uses fitted summaries, while log-score gains use the held-out evaluation split. Negative gains remain in the results.


In [4]:
queries = pd.read_csv(RUN / "study_c_queries.csv")
metadata = read_json(RUN / "study_c_metadata.json")
failures = read_json(RUN / "study_c_failures.json")
expected_queries = 4 if MODE == "smoke" else 40
expected_comparisons = 3 * expected_queries
assert len(queries) == expected_comparisons == metadata["query_temperature_count"]
assert set(queries["beta"]) == {3, 6, 9}
assert queries.groupby("beta").size().eq(expected_queries).all()
assert not queries.duplicated(["beta", "query_id"]).any()
assert metadata["split_sizes"] == {"representation": 600, "query": 40, "reference": 600, "evaluation": 557}
assert not failures and queries["status"].eq("ok").all(), "Inspect study_c_failures.json."
require_finite(queries, ["I1", "I2", "test_gain", "max_moment_error", "grid_gap_change", "grid_test_gain_change"])
protocol = read_json(RUN / "study_c_protocol.json")
assert queries["max_moment_error"].max() <= protocol["accuracy"]["moment_error_max"]
assert queries[["grid_gap_change", "grid_test_gain_change"]].to_numpy().max() <= protocol["accuracy"]["gap_and_test_gain_grid_change_max"]

with np.load(RUN / "study_c_arrays.npz", allow_pickle=False) as arrays:
    groups = {name: arrays[name + "_ids"] for name in metadata["split_sizes"]}
    assert all(len(ids) == metadata["split_sizes"][name] for name, ids in groups.items())
    all_ids = np.concatenate(list(groups.values()))
    assert len(all_ids) == len(np.unique(all_ids)) == 1797
    assert set(queries["query_id"]).issubset(set(groups["query"]))
display(pd.Series(metadata["split_sizes"], name="images").to_frame())
display(queries.groupby("beta")[["max_moment_error", "grid_gap_change", "grid_test_gain_change"]].max())
print(f"Verified {expected_comparisons} query-temperature comparisons.")


,images
representation,600
query,40
reference,600
evaluation,557


,max_moment_error,grid_gap_change,grid_test_gain_change
beta,,,
3,2.141737e-11,4.440892e-15,4.440892e-15
6,1.147404e-11,5.329071e-15,5.301315e-15
9,5.935907e-11,8.881784e-15,8.895662e-15


Verified 12 query-temperature comparisons.


## Held-out gains and reference summaries

A fitted information gap measures improvement for the fitted moments. It does not guarantee a positive held-out gain. The table retains every selected query and temperature, including any negative evaluation gains.

Queries share fitting and evaluation images. Their results are dependent, and writer identifiers are unavailable. The correlations below are descriptive benchmark summaries, without population p-values or independent-query uncertainty intervals. Four-query smoke correlations are especially unstable.


In [5]:
summaries = read_json(RUN / "study_c_summary.json")
summary = pd.DataFrame(summaries)
assert len(summary) == 3
assert summary["successful_queries"].eq(expected_queries).all()
display(summary[["beta", "successful_queries", "positive_gain_queries", "negative_gain_queries", "mean_gain", "median_gain", "min_gain", "max_gain"]])
display(queries[["beta", "query_id", "query_label", "I1", "I2", "test_gain", "model_score", "fit_ess", "test_ess"]])
associations = pd.DataFrame({row["beta"]: row["spearman"] for row in summaries}).T
associations.index.name = "beta"
display(associations)
show_run_figure("study_c_heldout.png")


,beta,successful_queries,positive_gain_queries,negative_gain_queries,mean_gain,median_gain,min_gain,max_gain
0,6,4,4,0,0.180597,0.166256,0.144543,0.245335
1,3,4,4,0,0.065040,0.053410,0.032924,0.120414
2,9,4,4,0,0.135087,0.065615,0.037651,0.371466


,beta,query_id,query_label,I1,I2,test_gain,model_score,fit_ess,test_ess
0,6,485,1,0.301587,0.133143,0.245335,166.223783,72.536377,36.324423
1,6,1579,0,1.538646,0.324890,0.181404,1405.640773,67.357518,68.713888
2,6,837,7,1.216707,0.190434,0.144543,484.421000,46.637613,45.884443
3,6,1200,7,0.994342,0.166610,0.151107,318.290413,29.207869,43.986202
4,3,485,1,0.048197,0.051498,0.062875,64.777569,341.781649,225.944128
5,3,1579,0,0.402691,0.147190,0.120414,188.566361,225.521613,219.375651
6,3,837,7,0.328517,0.079667,0.032924,101.188412,244.498989,227.805239
7,3,1200,7,0.218711,0.066549,0.043946,78.605848,270.776327,243.239609
8,9,485,1,0.795372,0.258228,0.371466,309.827128,19.442162,15.181929
9,9,1579,0,2.567092,0.285803,0.037651,5510.742063,38.712548,38.634046


,I1,I2,raw_second_norm,residual_second_norm,model_score,sandwich_wald
beta,,,,,,
6,-0.4,-0.4,-0.4,0.4,-0.4,-0.4
3,0.2,0.2,0.2,0.2,0.2,0.4
9,-0.8,-0.2,-0.8,0.8,-1.0,0.4


Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.


## A descriptive fixed-budget comparison

Each fitted summary selects half of the analyzed queries for the richer model. The always-simple and always-richer choices are endpoints. Selection uses only the fitting pool; evaluation uses held-out weighted log scores.

This comparison does not train or validate a deployment rule. In smoke mode it selects two queries; the paper comparison selects twenty. Similarity to the score-based summary is assessed for the primary temperature, not asserted across every setting.


In [6]:
policies = pd.read_csv(RUN / "study_c_policies.csv")
primary = policies.loc[policies["beta"].eq(6)].copy()
assert len(primary) == 8
assert primary.loc[primary["policy"].str.startswith("top_half_"), "selected"].eq(expected_queries // 2).all()
require_finite(primary, ["selected_mean_gain", "per_query_policy_gain"])
display(primary[["policy", "selected", "per_query_policy_gain", "selected_mean_gain", "negative_selected", "query_ids"]])
validation_path = RUN / "study_c_validation.json"
if validation_path.is_file():
    validation = read_json(validation_path)
    assert validation["passed"]
    display(pd.Series(validation["max_discrepancies"], name="saved-data check discrepancy").to_frame())
    print(validation["scope"])


,policy,selected,per_query_policy_gain,selected_mean_gain,negative_selected,query_ids
0,top_half_I1,2,0.081487,0.162974,0,1579;837
1,top_half_I2,2,0.081487,0.162974,0,1579;837
2,top_half_raw_second_norm,2,0.081487,0.162974,0,1579;837
3,top_half_residual_second_norm,2,0.099110,0.198221,0,485;1200
4,top_half_model_score,2,0.081487,0.162974,0,1579;837
5,top_half_sandwich_wald,2,0.083128,0.166256,0,1200;1579
6,always_vMF,0,0.000000,0.000000,0,NaN
7,always_FB,4,0.180597,0.180597,0,485;1579;837;1200


,saved-data check discrepancy
shared_pipeline_score_relative,1.710313e-12
shared_pipeline_wald_relative,3.726704e-13
unit_sphere,3.330669e-16
mean_match,0.000000e+00
raw_frobenius,3.330669e-16
residual_frobenius,3.386180e-15
empirical_logscore_identity,7.993606e-15
analytic_vmf_I1,5.329071e-15
analytic_vmf_partition,5.329071e-15
finer_partition,9.769963e-15


Saved-data integrity, independent identities, and numerical refinement; not population inference.
